# 인과 모델 기반 주별 원인 설명 연구

빈도 기반 방법을 제외하고, 인과 그래프 추정 방법과 그래프 기반 귀속을 비교한다.

실행: 저장소 루트에서 `uv run python tools/run_causal_study.py`.
총 12시간, 적합 한 건당 45분을 기본 한도로 사용한다. 이 노트북은 저장된 결과만 읽으며 모델을 다시 적합하지 않는다.

- 선택: 2023-07~2024-12, 학습은 2023-01 이전.
- 확인: 2025-01~2026-10, 학습은 2025-01 이전.
- 주지표: 인용 변수가 입력에 있는 주에서 top-3 후보 중 하나라도 답지와 일치한 비율.
- 커피는 CEPEA 헤알 가격이다. 예측 파이프라인 USD 타깃 성능과 구분한다.
- 두 구간은 이미 본 역사적 기간이다. 결과는 미사용 holdout 평가가 아니다.


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display
ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
BASE = ROOT / "data/07_reporting/causal_study"
# 특정 실행을 보려면 RUN에 그 실행 디렉터리를 직접 지정한다.
RUN = None
if RUN is None:
    pointer = BASE / "latest.json"
    if pointer.exists():
        latest = json.loads(pointer.read_text())
        value = latest if isinstance(latest, str) else latest.get("path", latest.get("run_dir"))
        RUN = Path(value) if value else None
        if RUN is not None and not RUN.is_absolute():
            RUN = ROOT / RUN
    if RUN is None:
        runs = sorted(BASE.glob("*/report.md"), key=lambda p: p.stat().st_mtime)
        if not runs:
            raise FileNotFoundError("먼저 tools/run_causal_study.py를 실행하세요.")
        RUN = runs[-1].parent
print("결과 디렉터리:", RUN)


## 비교 결과와 상태

실행 상태가 미완료인 모델은 0점 모델로 읽지 않는다. 전체 원자재 결과가 없는 후보는 평균 순위에서 제외한다.


In [ ]:
report = RUN / "report.md"
if report.exists():
    display(Markdown(report.read_text()))
else:
    print("실행 중: 아직 보고서가 없습니다.")
for name in ("summary.csv", "status.json"):
    path = RUN / name
    if path.exists():
        if path.suffix == ".csv":
            display(pd.read_csv(path))
        else:
            display(json.loads(path.read_text()))


## 해석 경계

- 같은 주 방향을 가정한 링크와 모델이 식별한 방향을 구분한다.
- 미정·교란 링크를 확정 부모로 바꾸지 않으며, 부모가 3개 미만이면 빈도나 상관 순위로 채우지 않는다.
- 검정 통계량은 인과효과가 아니다. 귀속은 가정한 그래프 아래의 모델 설명이다.
- 모델 비교는 귀속을 고정하고, ablation은 부모와 변경 축을 명시한다.
- 확인 구간의 짝지은 4주 블록 bootstrap 95% 구간이 0보다 클 때만 개선 확인으로 읽는다.
- VARLiNGAM은 결측을 받지 못해, 늦게 시작하는 변수의 시작 전 구간을 0으로 채운 연속 구간에서 같은 조건의 PCMCI 대조군(`matched_pcmci`)과 함께 읽는다.

공식 방법 문서: [Tigramite](https://jakobrunge.github.io/tigramite/), [VARLiNGAM](https://lingam.readthedocs.io/en/stable/reference/var_lingam.html).
